# DEE Test T³ — Versión diagnóstica con exportación de gráficos

Este notebook repite los tests del anterior con dos diferencias:

1. **Guarda todos los plots como PNG** en el directorio de trabajo
2. **Agrega barridos más finos** en k y N para diagnosticar la dispersión del 20.82% que apareció en el run inicial
3. **Descarga los gráficos como zip** al final para que los tengas en tu laptop

Los tests agregados son específicos para entender **por qué** la dispersión fue alta:
- Test A: barrido fino en k (8, 12, 18, 24, 30)
- Test B: N más grande (2000, 3000) para ver convergencia
- Test C: muestreo cuasi-uniforme (grilla perturbada) vs uniforme puro

---

In [ ]:
import os
import numpy as np
from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt

# Directorio para guardar plots
PLOT_DIR = 'plots_T3_diagnostico'
os.makedirs(PLOT_DIR, exist_ok=True)
print(f'Plots se guardarán en: {PLOT_DIR}/')

# Helper para guardar
def save_plot(name, fig=None, dpi=120):
    if fig is None:
        fig = plt.gcf()
    path = os.path.join(PLOT_DIR, f'{name}.png')
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  → guardado: {path}')

## 1. Funciones base (idénticas al notebook anterior)

In [ ]:
def periodic_distance_matrix(points, L=1.0):
    diff = points[:, None, :] - points[None, :, :]
    diff = diff - L * np.round(diff / L)
    return np.linalg.norm(diff, axis=-1)

def build_DEE_laplacian(points, k_neighbors=12, alpha=2.0, L=1.0):
    N = len(points)
    D_mat = periodic_distance_matrix(points, L)
    D_search = D_mat.copy()
    np.fill_diagonal(D_search, np.inf)
    neighbor_idx = np.argpartition(D_search, k_neighbors, axis=1)[:, :k_neighbors]
    
    rows, cols, vals = [], [], []
    for i in range(N):
        for j in neighbor_idx[i]:
            d = D_mat[i, j]
            if d > 0:
                w = 1.0 / d**alpha
                rows.append(i)
                cols.append(j)
                vals.append(w)
    
    W = csr_matrix((vals, (rows, cols)), shape=(N, N))
    W = (W + W.T) / 2
    degrees = np.array(W.sum(axis=1)).flatten()
    D_diag = diags(degrees)
    return D_diag - W, degrees.mean()

def get_eigenvalues(L_sparse, n_eig=25):
    try:
        eigs, _ = eigsh(L_sparse, k=n_eig, which='SM', tol=1e-8)
    except Exception:
        eigs_all = np.linalg.eigvalsh(L_sparse.toarray())
        eigs = eigs_all[:n_eig]
    return np.sort(eigs)

def dispersion_sexteto(eigs):
    s = eigs[1:7]
    return (s.max() - s.min()) / s.max()

print('Funciones base listas.')

## 2. Test base y plot espectro T³

In [ ]:
# Test base con los parámetros del resultado previo
rng = np.random.default_rng(42)
N = 1200
points = rng.uniform(0, 1, size=(N, 3))
L_T3, d_mean = build_DEE_laplacian(points, k_neighbors=12, alpha=2.0, L=1.0)
eigs_base = get_eigenvalues(L_T3, n_eig=25)

print(f'Base case N=1200, k=12, α=2.0:')
print(f'  Grado medio: {d_mean:.2f}')
print(f'  Dispersión sexteto: {dispersion_sexteto(eigs_base)*100:.2f}%')
print(f'  λ₇/λ₆ = {eigs_base[7]/eigs_base[6]:.3f}')

# Plot 1: espectro completo
fig, ax = plt.subplots(figsize=(10, 6))
eigs_norm = eigs_base / eigs_base[1]
ax.plot(np.arange(len(eigs_norm)), eigs_norm, 'o-', color='steelblue',
        markersize=10, linewidth=2, label='Grafo DEE (N=1200, k=12, α=2)')

# Valores teóricos T³
theoretical = []
for nx in range(-3, 4):
    for ny in range(-3, 4):
        for nz in range(-3, 4):
            theoretical.append(nx**2 + ny**2 + nz**2)
theoretical = np.sort(theoretical)[:25]
ax.plot(np.arange(len(theoretical)), theoretical, 'x', color='red',
        markersize=14, markeredgewidth=2.5, label='T³ analítico (λ/λ₁ = |n|²)')

# Sombreados para sexteto y bloques siguientes
ax.axvspan(0.5, 6.5, alpha=0.15, color='green', label='Sexteto esperado (|n|²=1)')
ax.axvspan(6.5, 18.5, alpha=0.08, color='orange', label='12-plete esperado (|n|²=2)')
ax.axvspan(18.5, 26.5, alpha=0.05, color='purple', label='8-plete esperado (|n|²=3)')

ax.set_xlabel('Índice del autovalor', fontsize=12)
ax.set_ylabel('λ / λ₁ (normalizado)', fontsize=12)
ax.set_title('Espectro del Laplaciano DEE en T³ — Run base', fontsize=13)
ax.legend(loc='upper left', fontsize=10)
ax.grid(alpha=0.3)
ax.set_ylim(-0.3, 4.5)
plt.tight_layout()
save_plot('01_espectro_base_N1200')
plt.show()

## 3. Test A: barrido fino en k (número de vecinos)

Hipótesis: con k=12 la cobertura angular es insuficiente para que el Laplaciano sea isotrópico en 3D. Valores k múltiplos de 3 (6, 12, 18, 24, 30) deberían dar mejor resultado según §3.7.

In [ ]:
k_values = [6, 12, 18, 24, 30]
disp_by_k = {}
eigs_by_k = {}

for k in k_values:
    rng = np.random.default_rng(42)
    points = rng.uniform(0, 1, size=(N, 3))
    L_mat, dm = build_DEE_laplacian(points, k_neighbors=k, alpha=2.0, L=1.0)
    eigs = get_eigenvalues(L_mat, n_eig=25)
    d = dispersion_sexteto(eigs)
    disp_by_k[k] = d
    eigs_by_k[k] = eigs
    print(f'k={k:2d}: disp sexteto = {d*100:5.2f}%, λ₇/λ₆ = {eigs[7]/eigs[6]:.3f}, grado medio = {dm:.2f}')

# Plot A1: dispersión vs k
fig, ax = plt.subplots(figsize=(10, 5))
ks = list(disp_by_k.keys())
ds = [disp_by_k[k]*100 for k in ks]
ax.plot(ks, ds, 'o-', markersize=12, linewidth=2, color='darkblue')
ax.axhline(10, linestyle='--', color='green', alpha=0.5, label='Umbral "bueno" (10%)')
ax.axhline(5, linestyle=':', color='darkgreen', alpha=0.5, label='Umbral "excelente" (5%)')
# Destacar múltiplos de 3
for k in [6, 12, 18, 24, 30]:
    ax.axvline(k, color='gray', alpha=0.2)
ax.set_xlabel('k (número de vecinos)', fontsize=12)
ax.set_ylabel('Dispersión relativa del sexteto [%]', fontsize=12)
ax.set_title('Dispersión del sexteto vs número de vecinos k', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
save_plot('02_dispersion_vs_k')
plt.show()

# Plot A2: espectros superpuestos para los k
fig, ax = plt.subplots(figsize=(11, 6))
cmap = plt.cm.viridis
for i, k in enumerate(k_values):
    eigs = eigs_by_k[k]
    eigs_norm = eigs / eigs[1]
    color = cmap(i / (len(k_values)-1))
    ax.plot(np.arange(len(eigs_norm[:20])), eigs_norm[:20], 'o-',
            color=color, markersize=8, linewidth=1.5,
            label=f'k={k}  (disp: {disp_by_k[k]*100:.1f}%)')

theoretical_20 = np.sort([nx**2+ny**2+nz**2 for nx in range(-3,4) for ny in range(-3,4) for nz in range(-3,4)])[:20]
ax.plot(np.arange(20), theoretical_20, 'x', color='red', markersize=14,
        markeredgewidth=2.5, label='T³ analítico')
ax.axvspan(0.5, 6.5, alpha=0.1, color='green')
ax.set_xlabel('Índice del autovalor', fontsize=12)
ax.set_ylabel('λ / λ₁', fontsize=12)
ax.set_title('Espectros para distintos k — el mejor k es el que más se acerca a las cruces rojas', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_ylim(-0.3, 4.5)
plt.tight_layout()
save_plot('03_espectros_por_k')
plt.show()

## 4. Test B: N más grande (con el mejor k encontrado)

Si la dispersión alta era por muestreo pobre, con N=2000-3000 debería mejorar. Si se estanca, el problema es del kernel, no del muestreo.

In [ ]:
# Usar el mejor k del test anterior
k_best = min(disp_by_k, key=disp_by_k.get)
print(f'Mejor k del test anterior: k={k_best} (dispersión {disp_by_k[k_best]*100:.2f}%)')
print()

N_values = [500, 800, 1200, 2000, 3000]
disp_by_N = {}
eigs_by_N = {}

for N_test in N_values:
    rng = np.random.default_rng(42)
    points = rng.uniform(0, 1, size=(N_test, 3))
    L_mat, dm = build_DEE_laplacian(points, k_neighbors=k_best, alpha=2.0, L=1.0)
    eigs = get_eigenvalues(L_mat, n_eig=25)
    d = dispersion_sexteto(eigs)
    disp_by_N[N_test] = d
    eigs_by_N[N_test] = eigs
    print(f'N={N_test:4d}: disp sexteto = {d*100:5.2f}%, λ₇/λ₆ = {eigs[7]/eigs[6]:.3f}')

# Plot B: convergencia con N
fig, ax = plt.subplots(figsize=(10, 5))
Ns = sorted(disp_by_N.keys())
ds = [disp_by_N[n]*100 for n in Ns]
ax.loglog(Ns, ds, 'o-', markersize=12, linewidth=2, color='darkred', label=f'Datos (k={k_best})')

# Líneas de referencia teóricas
N_arr = np.array(Ns)
ax.plot(N_arr, 30*(N_arr/500)**(-0.5), '--', color='gray', alpha=0.6, label='∝ N^(−1/2) (muestreo ideal)')
ax.plot(N_arr, 30*(N_arr/500)**(-0.33), ':', color='gray', alpha=0.6, label='∝ N^(−1/3)')

ax.axhline(10, linestyle='--', color='green', alpha=0.5, label='Umbral "bueno"')
ax.set_xlabel('N (número de nodos)', fontsize=12)
ax.set_ylabel('Dispersión del sexteto [%]', fontsize=12)
ax.set_title(f'Convergencia con N (k={k_best}, α=2)', fontsize=13)
ax.legend()
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
save_plot('04_convergencia_N')
plt.show()

## 5. Test C: muestreo cuasi-uniforme vs uniforme

Un muestreo aleatorio uniforme puro puede dejar vacíos locales grandes, especialmente en 3D. Una alternativa más isotrópica es una grilla regular perturbada: puntos en grilla regular + jitter pequeño. Esto aproxima el muestreo cuasi-uniforme que el documento menciona en §3.5.

In [ ]:
def grid_perturbed_sample(N_target, jitter=0.3, seed=42):
    """
    Genera puntos en grilla regular 3D con jitter.
    N_target se redondea al cubo perfecto más cercano.
    jitter: fracción del spacing de grilla.
    """
    rng = np.random.default_rng(seed)
    # Cubo más cercano
    n_side = round(N_target**(1/3))
    N_actual = n_side**3
    spacing = 1.0 / n_side
    
    coords = np.arange(n_side) * spacing + spacing/2
    grid_x, grid_y, grid_z = np.meshgrid(coords, coords, coords, indexing='ij')
    points = np.stack([grid_x.ravel(), grid_y.ravel(), grid_z.ravel()], axis=1)
    
    # Jitter
    points += rng.uniform(-jitter*spacing, jitter*spacing, size=points.shape)
    # Wrap a [0, 1)
    points = points % 1.0
    return points, N_actual

# Test con grilla perturbada
points_grid, N_actual = grid_perturbed_sample(1200, jitter=0.25, seed=42)
print(f'Grilla perturbada: n_side=10 → N={N_actual}, jitter=0.25·spacing')

L_grid, dm_grid = build_DEE_laplacian(points_grid, k_neighbors=k_best, alpha=2.0, L=1.0)
eigs_grid = get_eigenvalues(L_grid, n_eig=25)
d_grid = dispersion_sexteto(eigs_grid)
print(f'  Dispersión sexteto: {d_grid*100:.2f}%')
print(f'  λ₇/λ₆: {eigs_grid[7]/eigs_grid[6]:.3f}')

# Comparación
rng = np.random.default_rng(42)
points_uniform = rng.uniform(0, 1, size=(N_actual, 3))
L_uni, _ = build_DEE_laplacian(points_uniform, k_neighbors=k_best, alpha=2.0, L=1.0)
eigs_uni = get_eigenvalues(L_uni, n_eig=25)
d_uni = dispersion_sexteto(eigs_uni)

print(f'\nComparación N={N_actual}, k={k_best}:')
print(f'  Uniforme puro:      disp = {d_uni*100:5.2f}%')
print(f'  Grilla perturbada:  disp = {d_grid*100:5.2f}%')

# Plot C
fig, ax = plt.subplots(figsize=(11, 6))
eigs_grid_n = eigs_grid / eigs_grid[1]
eigs_uni_n = eigs_uni / eigs_uni[1]

ax.plot(np.arange(20), eigs_uni_n[:20], 'o-', color='steelblue',
        markersize=10, linewidth=2, label=f'Uniforme puro (disp {d_uni*100:.1f}%)')
ax.plot(np.arange(20), eigs_grid_n[:20], 's-', color='darkorange',
        markersize=10, linewidth=2, label=f'Grilla perturbada (disp {d_grid*100:.1f}%)')
theoretical_20 = np.sort([nx**2+ny**2+nz**2 for nx in range(-3,4) for ny in range(-3,4) for nz in range(-3,4)])[:20]
ax.plot(np.arange(20), theoretical_20, 'x', color='red', markersize=14,
        markeredgewidth=2.5, label='T³ analítico')
ax.axvspan(0.5, 6.5, alpha=0.15, color='green')
ax.set_xlabel('Índice del autovalor', fontsize=12)
ax.set_ylabel('λ / λ₁', fontsize=12)
ax.set_title(f'Muestreo uniforme vs grilla perturbada (N={N_actual}, k={k_best})', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_ylim(-0.3, 4.5)
plt.tight_layout()
save_plot('05_uniforme_vs_grilla')
plt.show()

## 6. Resumen diagnóstico y descarga

In [ ]:
print('='*65)
print('RESUMEN DIAGNÓSTICO — Test T³ DEE')
print('='*65)
print()
print(f'Run base (N=1200, k=12, α=2, uniforme):')
print(f'  Dispersión sexteto: {dispersion_sexteto(eigs_base)*100:.2f}%')
print(f'  Interpretación: señal de T³ presente pero con ruido alto')
print()
print('Barrido en k:')
for k in k_values:
    marker = ' ←' if k == k_best else ''
    print(f'  k={k:2d}: {disp_by_k[k]*100:5.2f}%{marker}')
print()
print('Convergencia con N (k óptimo):')
for n in sorted(disp_by_N.keys()):
    print(f'  N={n:4d}: {disp_by_N[n]*100:5.2f}%')
print()
print(f'Comparación muestreo (N={N_actual}, k={k_best}):')
print(f'  Uniforme puro:     {d_uni*100:5.2f}%')
print(f'  Grilla perturbada: {d_grid*100:5.2f}%')
print()
print('='*65)

# Diagnóstico automático
best_case_disp = min(
    min(disp_by_k.values()),
    min(disp_by_N.values()),
    d_grid, d_uni
) * 100

print(f'\nMejor dispersión observada en todos los tests: {best_case_disp:.2f}%')

if best_case_disp < 10:
    print('\n→ DIAGNÓSTICO: el sexteto T³ emerge claramente con parámetros adecuados.')
    print('  El run base tuvo parámetros subóptimos. DEE pasa el test T³.')
elif best_case_disp < 20:
    print('\n→ DIAGNÓSTICO: estructura T³ presente pero no limpia.')
    print('  Podría ser limitación computacional (N insuficiente para 3D) o')
    print('  necesitar refinamiento del kernel/vecindad. Investigar más.')
else:
    print('\n→ DIAGNÓSTICO: la emergencia del sexteto T³ no es robusta.')
    print('  Pasar esto al autor para revisión del modelo, no es defecto del test.')

In [ ]:
# Empaquetar todos los plots en un zip para descarga
import shutil
shutil.make_archive('plots_T3_diagnostico', 'zip', PLOT_DIR)
print(f'\nArchivo zip creado: plots_T3_diagnostico.zip')
print(f'Contiene {len(os.listdir(PLOT_DIR))} plots.')

# Descarga automática desde Colab
try:
    from google.colab import files
    files.download('plots_T3_diagnostico.zip')
    print('\n→ Iniciando descarga del zip...')
except ImportError:
    print('\n(Si no estás en Colab, el zip está en el directorio actual)')